In [35]:
import pandas as pd


In [36]:
tranco_dataset = pd.read_csv("tranco_VQPQN.csv", header=None)
print(tranco_dataset.shape)

(1000000, 2)


## Get SPF results

In [37]:
spf_results = pd.read_csv("spf_results.csv")
print(spf_results.shape)

C:\Users\Elizabeth\AppData\Local\Temp\ipykernel_8384\319167684.py:1: DtypeWarning: Columns (6,8) have mixed types. Specify dtype option on import or set low_memory=False.
  spf_results = pd.read_csv("spf_results.csv")


(1000000, 14)


## Get DMARC results

In [38]:
dmarc_results = pd.read_csv("dmarc_final.csv")
dmarc_columns= ['domain', 'dmarc_isPresent', "dmarc_policy", 
                "dmarc_sp", "dmarc_pct", "dmarc_rua", "dmarc_ruf",
                "dmarc_adkim", "dmarc_aspf", "dmarc_error", "dmarc_valid"]
dmarc_results.columns = dmarc_columns
print(dmarc_results.shape)

(1000000, 11)


## Get DKIM results

In [39]:
# empty for now
dkim_columns = ["row_index","domain","tranco_rank","ranking_tier","query_timestamp_utc","dkim_present",
                "matched_selector","key_algorithm","key_length_bits","revoked"]

dkim_results = pd.DataFrame(columns=dkim_columns)
print(dkim_results.shape)

(0, 10)


# Merge results

In [40]:
spf_dmarc_result_merge = pd.merge(spf_results, dmarc_results, on='domain', how='inner')
print("spf_dmarc merge", spf_dmarc_result_merge.shape)

spf_dmarc merge (1000000, 24)


In [41]:
spf_not_in_dmarc = (
    spf_results
    .merge(dmarc_results, on='domain', how='left', indicator=True)
    .query('_merge == "left_only"')
    .drop(columns=['_merge'])
)

In [42]:
spf_not_in_dmarc = spf_results[~spf_results['domain'].isin(dmarc_results['domain'])]
print(spf_not_in_dmarc)

Empty DataFrame
Columns: [domain, tranco_rank, ranking_tier, query_timestamp, spf_present, spf_record_count, spf_raw_record, policy_strictness, include_chain, dns_lookup_count, dns_lookup_limit_exceeded, multiple_spf_records, has_ptr_mechanism, error]
Index: []


In [46]:
all_result_merge = pd.merge(spf_dmarc_result_merge, dkim_results, on="domain", how="outer")
print("all result merge", all_result_merge.shape)

all result merge (1000000, 33)
